# Simple GAN — Unconditional Face Generation on CelebA
**Architecture:** DCGAN (Radford et al., 2015)  
**Dataset:** CelebA 64×64 · **Framework:** PyTorch · **Platform:** Google Colab

---
### Purpose
This notebook implements a **basic unconditional GAN** as a starting point before moving to the more complex AttGAN.

| | Simple GAN | AttGAN |
|---|---|---|
| Conditioning | None (random noise) | Attribute vector |
| Control | None | Edit specific facial attributes |
| Loss | BCE adversarial only | Adv + Classification + Reconstruction |
| Architecture | DCGAN | Encoder–Decoder + Discriminator |

---
1. `Runtime → Change runtime type → GPU (T4)`
2. Run cells in order

## Cell 1 — Clone repo & install

In [ ]:
!git clone https://github.com/YOUR_USERNAME/GAN_Project_DL.git
%cd GAN_Project_DL
!pip install -q -r requirements.txt

import torch, sys
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

## Cell 2 — Config & device

In [ ]:
import torch
from config import Config

cfg = Config()
cfg.EXPERIMENT_NAME = 'simple_gan'
cfg.__init__()    # creates results/simple_gan/ and checkpoints/simple_gan/

# Quick smoke test: reduce epochs
# cfg.N_EPOCHS = 5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Results will be saved to: {cfg.RESULTS_DIR}')

## Cell 3 — Load CelebA

Images are resized to 64×64 inside the training loop (the DCGAN architecture expects 64×64).

In [ ]:
from src.dataset import get_loaders

train_loader, test_loader = get_loaders(cfg)

imgs, attrs = next(iter(train_loader))
print(f'Image batch: {imgs.shape}')

## Cell 4 — Build models

**DCGAN architecture:**
```
noise z (100×1×1)
   │
   ▼  Generator (5× ConvTranspose2d)
fake image (3×64×64)
   │
   ▼  Discriminator (5× Conv2d)
real/fake scalar
```
Loss: **BCE** — `L_G = BCE(D(G(z)), 1)` · `L_D = BCE(D(x),1) + BCE(D(G(z)),0)`

In [ ]:
from src.simple_gan import build_simple_models

LATENT_DIM = 100
gen, dis = build_simple_models(latent_dim=LATENT_DIM, device=device)

## Cell 5 — Train

In [ ]:
from src.simple_gan import train_simple_gan

g_losses, d_losses = train_simple_gan(
    gen, dis, train_loader, cfg, device, latent_dim=LATENT_DIM
)

## Cell 6 — Loss curves

In [ ]:
from src.utils import plot_losses
plot_losses(g_losses, d_losses, cfg)

## Cell 7 — FID & DACID metrics

**FID** measures the Fréchet distance between Inception feature distributions of real vs. generated images — the standard GAN evaluation metric.

**DACID** (our custom metric) measures the L2 distance between the *mean* feature vectors — simpler and faster than FID, useful for quick comparisons.

In [ ]:
from src.metrics import compute_metrics_simple_gan

scores = compute_metrics_simple_gan(gen, test_loader, LATENT_DIM, cfg, device)
print(f"\nFID   : {scores['fid']}")
print(f"DACID : {scores['dacid']}")

# Save metrics.json so export_results.py can pick it up
import json
payload = {
    'experiment': cfg.EXPERIMENT_NAME,
    'model':      'SimpleGAN',
    'fid':        scores['fid'],
    'dacid':      scores['dacid'],
    'g_losses':   g_losses,
    'd_losses':   d_losses,
}
path = cfg.RESULTS_DIR / 'metrics.json'
with open(path, 'w') as f:
    json.dump(payload, f, indent=2)
print(f'Saved → {path}')

## Cell 8 — Save checkpoint

Save the trained models so you can reload them without re-training.

In [ ]:
import torch
ckpt_path = cfg.CHECKPOINT_DIR / 'simple_gan_final.pt'
torch.save({'gen': gen.state_dict(), 'dis': dis.state_dict()}, ckpt_path)
print(f'Checkpoint saved → {ckpt_path}')